# NetCDF to DGGS (H3) Conversion for Canada Climate Indices

This notebook converts NetCDF climate indices to DGGS H3 grids, determines optimal resolution, aggregates mean values, stores results in Zarr format, and generates a pydggsapi-compatible config.

## Workflow Overview

1. [Preprocessing and Aggregation](#step-1-preprocessing-and-aggregation): Load NetCDF files, analyze spatial resolution, map grids to H3, and aggregate values.
    - [Find NetCDF Files and Analyze Grid Resolution](#find-netcdf-files-and-analyze-grid-resolution)
    - [Map NetCDF Grid to H3 DGGS](#map-netcdf-grid-to-h3-dggs)
    - [Caching Grid Mapping](#caching-grid-mapping)
    - [Process and Aggregate NetCDF Data](#process-and-aggregate-netcdf-data)
    - [Aggregate to Lower Resolutions](#aggregate-to-lower-resolutions)
2. [Store Zarr Attributes](#step-2-store-zarr-attributes-for-variable-metadata): Save variable metadata as Zarr attributes for each resolution group.
3. [Scrape and Integrate Extra Metadata](#step-3-scrape-and-integrate-extra-metadata): Optionally scrape climatedata.ca for additional variable metadata.
4. [Compute Spatial Bounding Box](#step-4-compute-spatial-bounding-box): Dynamically determine spatial extent from NetCDF files.
5. [Build and Save DGGS Collection JSON](#step-5-build-and-save-dggs-collection-json-final-config): Generate and save the pydggsapi config using all dynamic information.

---


In [1]:
# Imports
import os
import glob
import xarray as xr
import numpy as np
import pandas as pd
import h3
import zarr
import tempfile
from tqdm.notebook import tqdm
import requests
import json
from bs4 import BeautifulSoup
import datetime


### Step 1: Preprocessing and Aggregation
#### Find NetCDF Files and Analyze Grid Resolution

In [ ]:
# Configurable indices and base directory
CLIMATE_INDICES = ["tx_max", "ice_days", "frost_days", "prcptot"]
BASE_DATA_DIR = "../data"  # Adjust as needed
CACHE_DIR = os.environ.get("CLIMATE_CACHE_DIR", "./cache")  # Configurable cache directory
os.makedirs(CACHE_DIR, exist_ok=True)

def find_netcdf_files(base_dir, indices):
    files = []
    for root, _, fnames in os.walk(base_dir):
        for fname in fnames:
            if fname.endswith(".nc") and any(idx in fname for idx in indices):
                files.append(os.path.join(root, fname))
    return files

nc_files = find_netcdf_files(BASE_DATA_DIR, CLIMATE_INDICES)
print(f"Found {len(nc_files)} NetCDF files.")


#### Analyze Spatial Resolution

In [ ]:
if not nc_files:
    print("No NetCDF files found in the directory. Please check BASE_DATA_DIR and file availability.")
    # Optionally, raise an error or skip further steps
else:
    resolutions = []
    for f in nc_files:
        ds = xr.open_dataset(f)
        lat = ds["lat"] if "lat" in ds else ds["latitude"]
        lon = ds["lon"] if "lon" in ds else ds["longitude"]
        # Estimate resolution (in degrees)
        lat_res = float(np.abs(lat[1] - lat[0]))
        lon_res = float(np.abs(lon[1] - lon[0]))
        # Approximate km using haversine formula
        mean_lat = float(lat.mean())
        earth_radius_km = 6371.0
        lat_km = lat_res * (np.pi/180) * earth_radius_km
        lon_km = lon_res * (np.pi/180) * earth_radius_km * np.cos(mean_lat * np.pi/180)
        resolutions.append((f, lat_km, lon_km))
        ds.close()

    # Find best (finest) resolution
    best_file, best_lat_km, best_lon_km = min(resolutions, key=lambda x: min(x[1], x[2]))
    print(f"Best spatial resolution: {best_lat_km:.2f} km x {best_lon_km:.2f} km from {best_file}")


#### Map NetCDF Grid to H3 DGGS

In [ ]:
H3_RESOLUTION = None
for res in range(15):
    h3_edge_km = h3.edge_length(res, unit="km")
    if h3_edge_km <= min(best_lat_km, best_lon_km):
        H3_RESOLUTION = res
        break
if H3_RESOLUTION is None:
    H3_RESOLUTION = 15  # fallback to max
print(f"Selected H3 resolution: {H3_RESOLUTION} (edge ~{h3.edge_length(H3_RESOLUTION, unit='km'):.2f} km)")


#### Caching Grid Mapping

In [ ]:
def cache_grid_to_h3(lat, lon, resolution, cache_dir=CACHE_DIR):
    cache_file = os.path.join(cache_dir, f"grid2h3_{resolution}_{hash(tuple(lat))}_{hash(tuple(lon))}.npz")
    if os.path.exists(cache_file):
        data = np.load(cache_file)
        return data["h3_indices"]
    h3_indices = np.empty((len(lat), len(lon)), dtype=object)
    for i, la in enumerate(lat):
        for j, lo in enumerate(lon):
            h3_indices[i, j] = h3.latlng_to_cell(float(la), float(lo), resolution)
    np.savez_compressed(cache_file, h3_indices=h3_indices)
    return h3_indices


#### Process and Aggregate NetCDF Data

In [ ]:
for f in nc_files:
    ds = xr.open_dataset(f)
    for var in CLIMATE_INDICES:
        if var not in ds:
            continue
        lat = ds["lat"] if "lat" in ds else ds["latitude"]
        lon = ds["lon"] if "lon" in ds else ds["longitude"]
        h3_indices = cache_grid_to_h3(lat, lon, H3_RESOLUTION)
        # Aggregate mean values for each h3 index and time
        time_dim = ds["time"] if "time" in ds else ds["year"]
        values = ds[var].values
        # Flatten grid and group by h3 index
        records = []
        for t_idx, t_val in enumerate(time_dim):
            grid_vals = values[t_idx] if values.ndim == 3 else values
            for i in range(len(lat)):
                for j in range(len(lon)):
                    h3_id = h3_indices[i, j]
                    records.append({
                        "h3_id": h3_id,
                        "datetime": pd.to_datetime(str(t_val)),
                        var: float(grid_vals[i, j])
                    })
        df = pd.DataFrame(records)
        # Group by h3_id and datetime, aggregate mean
        df = df.groupby(["h3_id", "datetime"])[var].mean().reset_index()
        # Save intermediate results for later aggregation
        temp_file = os.path.join(CACHE_DIR, f"{var}_h3_{H3_RESOLUTION}.parquet")
        df.to_parquet(temp_file)
    ds.close()


#### Aggregate to Lower Resolutions

In [ ]:
def aggregate_to_lower_res(df, start_res, min_res):
    dfs = {start_res: df}
    for res in range(start_res-1, min_res-1, -1):
        df_res = df.copy()
        df_res["h3_id"] = df_res["h3_id"].apply(lambda h: h3.cell_to_parent(h, res))
        df_res = df_res.groupby(["h3_id", "datetime"]).mean().reset_index()
        dfs[res] = df_res
    return dfs


### Step 3: Scrape and Integrate Extra Metadata

In [ ]:
def get_variable_metadata(indices, cache_dir=CACHE_DIR):
    cache_file = os.path.join(cache_dir, "climate_metadata.json")
    if os.path.exists(cache_file):
        with open(cache_file, "r") as f:
            return json.load(f)
    base_url = "https://climatedata.ca/variables/"
    metadata = {}
    for idx in indices:
        url = f"{base_url}{idx}/"
        try:
            resp = requests.get(url)
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, "html.parser")
                desc = ""
                unit = ""
                # Try to find description
                desc_tag = soup.find("meta", attrs={"name": "description"})
                if desc_tag and desc_tag.get("content"):
                    desc = desc_tag["content"]
                else:
                    # Fallback: look for <p> or <div> with description
                    p_tags = soup.find_all("p")
                    for p in p_tags:
                        if "description" in p.text.lower():
                            desc = p.text.strip()
                            break
                # Try to find unit
                unit_tag = soup.find("span", class_="unit")
                if unit_tag:
                    unit = unit_tag.text.strip()
                else:
                    # Fallback: search for 'unit' in text
                    for tag in soup.find_all(True):
                        if "unit" in tag.text.lower():
                            unit = tag.text.strip()
                            break
                metadata[idx] = {"description": desc, "unit": unit, "url": url}
            else:
                metadata[idx] = {"description": "", "unit": "", "url": url}
        except Exception as e:
            metadata[idx] = {"description": "", "unit": "", "url": url}
    with open(cache_file, "w") as f:
        json.dump(metadata, f)
    return metadata


In [ ]:
variable_metadata = get_variable_metadata(CLIMATE_INDICES)
print("Variable metadata:", variable_metadata)


### Step 2: Store Zarr Attributes for Variable Metadata

In [ ]:
zarr_store_path = os.path.join(CACHE_DIR, "climate_dggs.zarr")
root = zarr.open_group(zarr_store_path, mode="w")
for var in CLIMATE_INDICES:
    temp_file = os.path.join(CACHE_DIR, f"{var}_h3_{H3_RESOLUTION}.parquet")
    df = pd.read_parquet(temp_file)
    dfs = aggregate_to_lower_res(df, H3_RESOLUTION, 0)
    for res, df_res in dfs.items():
        grp = root.require_group(f"res{res}")
        arr = grp.create_array(var, shape=df_res[var].shape, dtype="float32", chunks=(10000,))
        arr[:] = df_res[var].values
        h3_id_arr = grp.create_array(f"h3_id_{var}", shape=df_res["h3_id"].shape, dtype="S15")
        h3_id_arr[:] = df_res["h3_id"].values
        dt_arr = grp.create_array(f"datetime_{var}", shape=df_res["datetime"].shape, dtype="S25")
        dt_arr[:] = df_res["datetime"].values.astype("str")
        # Add metadata as attributes
        if var in variable_metadata:
            arr.attrs["description"] = variable_metadata[var]["description"]
            arr.attrs["unit"] = variable_metadata[var]["unit"]
            arr.attrs["source_url"] = variable_metadata[var]["url"]

print(f"Zarr store created at {zarr_store_path}")


### Step 4: Compute Spatial Bounding Box

In [ ]:
def compute_spatial_bbox(nc_files):
    min_lat, max_lat, min_lon, max_lon = None, None, None, None
    for f in nc_files:
        ds = xr.open_dataset(f)
        lat = ds["lat"] if "lat" in ds else ds["latitude"]
        lon = ds["lon"] if "lon" in ds else ds["longitude"]
        lat_min, lat_max = float(lat.min()), float(lat.max())
        lon_min, lon_max = float(lon.min()), float(lon.max())
        if min_lat is None or lat_min < min_lat:
            min_lat = lat_min
        if max_lat is None or lat_max > max_lat:
            max_lat = lat_max
        if min_lon is None or lon_min < min_lon:
            min_lon = lon_min
        if max_lon is None or lon_max > max_lon:
            max_lon = lon_max
        ds.close()
    return [min_lon, min_lat, max_lon, max_lat]


In [ ]:
spatial_bbox = compute_spatial_bbox(nc_files)


#### Compute Temporal Range

In [ ]:
def compute_temporal_range(nc_files):
    min_time, max_time = None, None
    for f in nc_files:
        ds = xr.open_dataset(f)
        if "time" in ds:
            times = ds["time"].values
        elif "year" in ds:
            times = ds["year"].values
        else:
            ds.close()
            continue
        t_min, t_max = pd.to_datetime(times).min(), pd.to_datetime(times).max()
        if min_time is None or t_min < min_time:
            min_time = t_min
        if max_time is None or t_max > max_time:
            max_time = t_max
        ds.close()
    return [str(min_time), str(max_time)]


In [ ]:
temporal_range = compute_temporal_range(nc_files)
print("Temporal range:", temporal_range)


### Step 5: Build and Save DGGS Collection JSON (Final Config)

In [ ]:
def build_dggs_collection_json(best_lat_km, best_lon_km, temporal_range, variables, zarr_store_path, spatial_bbox):
    today = datetime.date.today().isoformat()
    collection_json = {
        "collections": {
            "canada-climatedata": {
                "title": "Canada Climate Indices DGGS Collection",
                "description": "Climate indices mapped to H3 DGGS cells, aggregated from NetCDF sources provided by ClimateData.ca.",
                "attribution": f"ClimateData.ca [Accessed on {today}]",
                "extent": {
                    "spatial": {
                        "bbox": [spatial_bbox],
                        "crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"
                    },
                    "temporal": {
                        "interval": [temporal_range],
                        "trs": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian"
                    }
                },
                "links": [
                    {
                        "rel": "about",
                        "href": "https://climatedata.ca/",
                        "type": "text/html",
                        "title": "ClimateData.ca - Canadian Climate Data Portal",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "about",
                        "href": "https://donneesclimatiques.ca/",
                        "type": "text/html",
                        "title": "Portail canadien des données climatiques",
                        "hreflang": "fr-CA"
                    },
                    {
                        "rel": "license",
                        "href": "https://creativecommons.org/licenses/by/4.0/legalcode",
                        "type": "text/html",
                        "title": "Creative Commons Attribution International (CC BY)",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "terms-of-service",
                        "href": "https://climatedata.ca/about/legal/terms/",
                        "type": "text/html",
                        "title": "ClimateData.ca Terms of Service",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "describedby",
                        "href": "https://climatedata.ca/variables/",
                        "type": "text/html",
                        "title": "ClimateData.ca Variables Documentation",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "convertedfrom",
                        "href": "https://climatedata.ca/",
                        "type": "text/html",
                        "title": "Original ClimateData.ca Dataset Portal",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "via",
                        "href": "https://github.com/crim-ca/ogc-dggs/tree/main/canada-climate",
                        "type": "text/html",
                        "title": "Source code employed to generate DGGS data from reference features",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "cite-as",
                        "href": "https://hirondelle.crim.ca/dggs-api/collections/canada-climatedata",
                        "type": "application/json",
                        "title": "Canada Climate Indices as DGGS Collection",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "sponsored",
                        "href": "https://climatedata.ca/about/citing-climatedata-ca/",
                        "type": "text/html",
                        "title": "ClimateData.ca Sponsorship and Funding",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "author",
                        "href": "https://crim.ca/fr/",
                        "type": "text/html",
                        "hreflang": "fr-CA",
                        "title": "Centre de recherche informatique de Montréal (CRIM)"
                    },
                    {
                        "rel": "author",
                        "href": "https://crim.ca/en/",
                        "type": "text/html",
                        "hreflang": "en-CA",
                        "title": "Centre de recherche informatique de Montréal (CRIM)"
                    }
                ],
                "collection_provider": {
                    "providerId": "canada-climatedata",
                    "dggrsId": "H3",
                    "min_refinement_level": 0,
                    "max_refinement_level": 8,
                    "datasource_id": "canada-climatedata"
                }
            }
        },
        "collection_providers": {
            "canada-climatedata": {
                "classname": "zarr_collection_provider.ZarrCollectionProvider",
                "datasources": {
                    "canada-climatedata": {
                        "filepath": zarr_store_path,
                        "id_col": "h3_id",
                        "datetime_col": "datetime",
                        "data_cols": variables
                    }
                }
            }
        },
        "dggrs": {
            "H3": {
                "title": "DGGRS H3",
                "description": "H3 hexagonal hierarchical geospatial indexing system developed by Uber.",
                "crs": "wgs84",
                "shapeType": "hexagon",
                "uri": "https://www.opengis.net/def/dggrs/OGC/1.0/H3",
                "definition_link": "https://www.opengis.net/def/dggrs/OGC/1.0/H3",
                "defaultDepth": 0,
                "classname": "h3_dggrs_provider.H3Provider"
            }
        }
    }
    return collection_json

collection_json = build_dggs_collection_json(
    best_lat_km,
    best_lon_km,
    temporal_range,
    CLIMATE_INDICES,
    zarr_store_path,
    spatial_bbox
)
with open(os.path.join(CACHE_DIR, "dggs-collection.json"), "w") as f:
    json.dump(collection_json, f, indent=2)
